# Chapter 7 — Training Loop Anatomy (Practice)

Work through these exercises **after reading** `notes/ch07-training-loop-anatomy.md`.

Each exercise states the *decision you're practicing*, gives a stub cell to fill in, and is followed by a pre-written **verification cell** — run it to grade yourself. Hand-write your answers in your working copy under `solutions/` (this `template/` copy stays pristine), and don't peek at `solved/` until the verification passes or you're genuinely stuck.

In [1]:
# ============================================================
# TOPIC: The training loop — order, optimizers, schedules, clipping, checkpoints
# MATH:  Adam: m_t=b1*m+(1-b1)g, v_t=b2*v+(1-b2)g^2, theta -= lr*m_hat/(sqrt(v_hat)+eps)
# REF:   B00 ch07 notes — training-loop-anatomy
# ============================================================

# --- Imports ---
import copy
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Reproducibility & device ---
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version : {torch.__version__}")
print(f"device        : {device}  (every exercise here runs fine on CPU)")

torch version : 2.13.0
device        : cpu  (every exercise here runs fine on CPU)


## Exercise 1 — Fix the Scrambled Loop

`scrambled_step` below has the seven steps in the WRONG order (clip before backward, step before clip, zero_grad after backward instead of before). Read the notes §2 dependency chain, then write `correct_step` with the right order — and watch the loss trajectory actually decrease.

**Decision you're practicing:** the seven-step dependency chain — each step needs something the previous step produced.

In [2]:
torch.manual_seed(0)
tiny_model = nn.Linear(4, 1)
optimizer = torch.optim.SGD(tiny_model.parameters(), lr=0.1)
probe_inputs = torch.randn(8, 4)
probe_targets = torch.randn(8, 1)

def scrambled_step(model, opt, inputs, targets):
    """The WRONG order — do not fix this one, just observe it."""
    outputs = model(inputs)
    loss = F.mse_loss(outputs, targets)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # clip before backward!
    opt.step()                                                          # step before backward!
    loss.backward()
    opt.zero_grad(set_to_none=True)                                    # zero AFTER backward!
    return loss.item()

scrambled_losses = [scrambled_step(tiny_model, optimizer, probe_inputs, probe_targets) for _ in range(20)]
print(f"scrambled loop losses (first 3, last 3): {scrambled_losses[:3]} ... {scrambled_losses[-3:]}")

def correct_step(model, opt, inputs, targets):
    opt.zero_grad(set_to_none=True)                       # 1. clear the bowl — BEFORE backward
    outputs = model(inputs)                               # 2. forward
    loss = F.mse_loss(outputs, targets)                    # 3. measure
    loss.backward()                                        # 4. work backward — fills .grad
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # 5. clip — AFTER backward
    opt.step()                                              # 6. apply the fix
    return loss.item()

torch.manual_seed(0)
fixed_model = nn.Linear(4, 1)
fixed_optimizer = torch.optim.SGD(fixed_model.parameters(), lr=0.1)
fixed_losses = [correct_step(fixed_model, fixed_optimizer, probe_inputs, probe_targets) for _ in range(20)]
print(f"correct loop losses (first 3, last 3):   {fixed_losses[:3]} ... {fixed_losses[-3:]}")

scrambled loop losses (first 3, last 3): [2.6633565425872803, 2.6633565425872803, 2.6633565425872803] ... [2.6633565425872803, 2.6633565425872803, 2.6633565425872803]
correct loop losses (first 3, last 3):   [2.6633565425872803, 2.4307916164398193, 2.239898443222046] ... [1.5063152313232422, 1.4874961376190186, 1.4696695804595947]


**Verification**

In [3]:
# --- Verification: Exercise 1 ---
torch.manual_seed(1)
verify_model = nn.Linear(4, 1)
verify_optimizer = torch.optim.SGD(verify_model.parameters(), lr=0.1)
verify_inputs = torch.randn(8, 4)
verify_targets = torch.randn(8, 1)

losses = [correct_step(verify_model, verify_optimizer, verify_inputs, verify_targets) for _ in range(30)]
assert losses[0] is not None, "implement correct_step first"
assert losses[-1] < losses[0] * 0.5, f"loss should drop substantially: {losses[0]:.4f} -> {losses[-1]:.4f}"
# monotonic-ish: on this simple convex problem, correct order should mostly decrease
worsened_steps = sum(1 for i in range(1, len(losses)) if losses[i] > losses[i-1] * 1.5)
assert worsened_steps <= 2, f"too many big jumps ({worsened_steps}) — order likely still wrong somewhere"
print(f"loss: {losses[0]:.4f} -> {losses[-1]:.4f} over {len(losses)} correctly-ordered steps ✓")
print("Exercise 1 passed ✓")

loss: 0.5154 -> 0.0556 over 30 correctly-ordered steps ✓
Exercise 1 passed ✓


## Exercise 2 — AdamW vs Adam: Watching the Decoupling

Reproduce the notes §4 dry-run yourself: a single weight, **zero real gradient**, only `weight_decay` acting. Track both trajectories for 10 steps and confirm AdamW decays **geometrically** (`w *= (1 - lr*weight_decay)` every step) while Adam's L2-coupled decay does not match that clean ratio.

**Decision you're practicing:** why AdamW ≠ Adam even at identical hyperparameters — decoupled vs gradient-coupled weight decay.

In [4]:
def decay_only_trajectory(optimizer_cls, lr, weight_decay, num_steps):
    """Track w over num_steps, feeding a ZERO gradient every step (isolates weight decay)."""
    w = torch.tensor([2.0], requires_grad=True)
    optimizer = optimizer_cls([w], lr=lr, weight_decay=weight_decay)
    trace = []
    for _ in range(num_steps):
        w.grad = torch.zeros(1)          # no "real" gradient signal — pure decay test
        optimizer.step()
        trace.append(round(w.item(), 4))
    return trace

adam_trace = decay_only_trajectory(torch.optim.Adam, lr=0.1, weight_decay=0.5, num_steps=10)
adamw_trace = decay_only_trajectory(torch.optim.AdamW, lr=0.1, weight_decay=0.5, num_steps=10)
print(f"Adam  (L2-in-grad): {adam_trace}")
print(f"AdamW (decoupled) : {adamw_trace}")

geometric_ratio = 1 - 0.1 * 0.5   # = 0.95, the expected AdamW per-step multiplier
print(f"\nexpected AdamW ratio each step: {geometric_ratio}")
print(f"AdamW step 0→1 ratio: {adamw_trace[1] / adamw_trace[0]:.4f}   (should be ≈ {geometric_ratio})")

Adam  (L2-in-grad): [1.9, 1.8002, 1.7006, 1.6015, 1.503, 1.4051, 1.3082, 1.2123, 1.1177, 1.0246]
AdamW (decoupled) : [1.9, 1.805, 1.7147, 1.629, 1.5476, 1.4702, 1.3967, 1.3268, 1.2605, 1.1975]

expected AdamW ratio each step: 0.95
AdamW step 0→1 ratio: 0.9500   (should be ≈ 0.95)


**Verification**

In [5]:
# --- Verification: Exercise 2 ---
adam_trace = decay_only_trajectory(torch.optim.Adam, lr=0.1, weight_decay=0.5, num_steps=10)
adamw_trace = decay_only_trajectory(torch.optim.AdamW, lr=0.1, weight_decay=0.5, num_steps=10)
assert adam_trace and adamw_trace, "implement decay_only_trajectory first"
assert len(adam_trace) == 10 and len(adamw_trace) == 10

# AdamW must match the clean geometric ratio (1 - lr*wd) after the first step
expected_ratio = 1 - 0.1 * 0.5
for i in range(1, len(adamw_trace)):
    actual_ratio = adamw_trace[i] / adamw_trace[i - 1]
    assert abs(actual_ratio - expected_ratio) < 0.01, \
        f"AdamW step {i}: ratio {actual_ratio:.4f}, expected ≈{expected_ratio}"

# Adam's decay must NOT follow that same clean geometric ratio (it's coupled/adaptive)
adam_ratios = [adam_trace[i] / adam_trace[i-1] for i in range(1, len(adam_trace))]
assert max(abs(r - expected_ratio) for r in adam_ratios) > 0.01, \
    "Adam's trajectory looks geometric — expected the adaptive normalization to distort it"

# both must still be decaying overall
assert adam_trace[-1] < adam_trace[0] and adamw_trace[-1] < adamw_trace[0]
print(f"AdamW followed the clean {expected_ratio} ratio; Adam's decay was distorted by 1/sqrt(v_hat) ✓")
print("Exercise 2 passed ✓")

AdamW followed the clean 0.95 ratio; Adam's decay was distorted by 1/sqrt(v_hat) ✓
Exercise 2 passed ✓


## Exercise 3 — Split Parameter Groups by ndim

Implement `build_param_groups(model, weight_decay)`: every parameter with `ndim >= 2` (weight matrices) gets the given `weight_decay`; every 1-D parameter (biases, LayerNorm gamma/beta) gets `weight_decay=0.0`.

**Decision you're practicing:** the `param_groups` convention — decay matrices, exempt biases and norm scale/shift.

In [6]:
class TinyTransformerBlock(nn.Module):
    def __init__(self, dim=8):
        super().__init__()
        self.linear = nn.Linear(dim, dim)      # weight: 2-D, bias: 1-D
        self.norm = nn.LayerNorm(dim)          # gamma, beta: both 1-D

def build_param_groups(model, weight_decay):
    """Return [{"params": [...], "weight_decay": weight_decay},
               {"params": [...], "weight_decay": 0.0}] split by param.ndim."""
    decay_params, no_decay_params = [], []
    for name, param in model.named_parameters():
        if param.ndim >= 2:
            decay_params.append(param)
        else:
            no_decay_params.append(param)
        print(f"  {name:20s} ndim={param.ndim} → {'DECAY' if param.ndim >= 2 else 'no decay'}")
    return [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]

block = TinyTransformerBlock()
groups = build_param_groups(block, weight_decay=0.1)
optimizer = torch.optim.AdamW(groups, lr=1e-3)
print(f"\ngroup 0 (decay={optimizer.param_groups[0]['weight_decay']}): {len(optimizer.param_groups[0]['params'])} tensors")
print(f"group 1 (decay={optimizer.param_groups[1]['weight_decay']}): {len(optimizer.param_groups[1]['params'])} tensors")

  linear.weight        ndim=2 → DECAY
  linear.bias          ndim=1 → no decay
  norm.weight          ndim=1 → no decay
  norm.bias            ndim=1 → no decay

group 0 (decay=0.1): 1 tensors
group 1 (decay=0.0): 3 tensors


**Verification**

In [7]:
# --- Verification: Exercise 3 ---
block = TinyTransformerBlock()
groups = build_param_groups(block, weight_decay=0.1)
assert groups is not None, "fill in the stub above first"
assert len(groups) == 2, "should return exactly two groups"

decay_group = next(g for g in groups if g["weight_decay"] == 0.1)
no_decay_group = next(g for g in groups if g["weight_decay"] == 0.0)

assert len(decay_group["params"]) == 1, "only linear.weight is 2-D"
assert decay_group["params"][0].shape == (8, 8)
assert len(no_decay_group["params"]) == 3, "linear.bias, norm.weight, norm.bias are all 1-D"
assert all(p.ndim == 1 for p in no_decay_group["params"])

# the optimizer must actually respect it: run steps and confirm the no-decay group barely
# moves under decay-only pressure while decay group shrinks (isolate with zero real grad)
optimizer = torch.optim.AdamW(groups, lr=0.1)
bias_before = block.linear.bias.clone()
weight_before = block.linear.weight.clone()
for _ in range(5):
    for p in block.parameters():
        p.grad = torch.zeros_like(p)
    optimizer.step()
assert torch.equal(block.linear.bias, bias_before), "bias has weight_decay=0 and zero grad — must NOT move"
assert not torch.equal(block.linear.weight, weight_before), "weight (2-D, decayed) should have shrunk"
print("bias (no decay, zero grad) unchanged; weight (decayed) shrank ✓")
print("Exercise 3 passed ✓")

  linear.weight        ndim=2 → DECAY
  linear.bias          ndim=1 → no decay
  norm.weight          ndim=1 → no decay
  norm.bias            ndim=1 → no decay
bias (no decay, zero grad) unchanged; weight (decayed) shrank ✓
Exercise 3 passed ✓


## Exercise 4 — Warmup + Cosine LR, From the Formula

Implement `warmup_cosine_lr(step, warmup_steps, total_steps, base_lr)` exactly per notes §6, then wrap it in a `LambdaLR` and confirm PyTorch's own `optimizer.param_groups[0]["lr"]` matches your formula at several checkpoints.

**Decision you're practicing:** the warmup-then-cosine shape, and how a `LambdaLR` multiplier turns a plain function into a working scheduler.

In [8]:
def warmup_cosine_lr(step, warmup_steps, total_steps, base_lr):
    """Linear warmup to base_lr, then cosine decay to 0 by total_steps."""
    if step < warmup_steps:
        return base_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * base_lr * (1 + math.cos(math.pi * progress))

BASE_LR, WARMUP, TOTAL = 1e-3, 10, 100
checkpoints = [0, 5, 10, 20, 55, 100]
manual_lrs = [warmup_cosine_lr(s, WARMUP, TOTAL, BASE_LR) for s in checkpoints]
for step, lr in zip(checkpoints, manual_lrs):
    print(f"step {step:3d}: lr = {lr:.6f}")

# wire it into a real LambdaLR: the lambda returns a MULTIPLIER on base_lr, so divide it out
model_for_sched = nn.Linear(2, 2)
sched_optimizer = torch.optim.SGD(model_for_sched.parameters(), lr=BASE_LR)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    sched_optimizer, lr_lambda=lambda s: warmup_cosine_lr(s, WARMUP, TOTAL, BASE_LR) / BASE_LR
)
pytorch_lrs = []
for step in range(TOTAL + 1):
    if step in checkpoints:
        pytorch_lrs.append(sched_optimizer.param_groups[0]["lr"])
    sched_optimizer.step()
    scheduler.step()
print(f"\nPyTorch LambdaLR at the same checkpoints: {[round(v, 6) for v in pytorch_lrs]}")

step   0: lr = 0.000000
step   5: lr = 0.000500
step  10: lr = 0.001000
step  20: lr = 0.000970
step  55: lr = 0.000500
step 100: lr = 0.000000

PyTorch LambdaLR at the same checkpoints: [0.0, 0.0005, 0.001, 0.00097, 0.0005, 0.0]


**Verification**

In [9]:
# --- Verification: Exercise 4 ---
manual_lrs = [warmup_cosine_lr(s, WARMUP, TOTAL, BASE_LR) for s in checkpoints]
assert manual_lrs[0] is not None, "implement warmup_cosine_lr first"
assert abs(manual_lrs[0] - 0.0) < 1e-9, "step 0 should be lr=0 (start of warmup)"
assert abs(manual_lrs[2] - BASE_LR) < 1e-9, "step 10 (end of warmup) should hit exactly base_lr"
assert abs(manual_lrs[4] - BASE_LR/2) < 1e-6, "step 55 is the schedule's halfway point → base_lr/2"
assert abs(manual_lrs[5] - 0.0) < 1e-6, "step 100 (end of training) should decay to ~0"
assert manual_lrs[1] < manual_lrs[2], "warmup should be monotonically increasing"
assert manual_lrs[3] < manual_lrs[2], "cosine decay should be decreasing after warmup"

# cross-check against a real LambdaLR
model_for_sched = nn.Linear(2, 2)
sched_optimizer = torch.optim.SGD(model_for_sched.parameters(), lr=BASE_LR)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    sched_optimizer, lr_lambda=lambda s: warmup_cosine_lr(s, WARMUP, TOTAL, BASE_LR) / BASE_LR
)
pytorch_lrs = []
for step in range(TOTAL + 1):
    if step in checkpoints:
        pytorch_lrs.append(sched_optimizer.param_groups[0]["lr"])
    sched_optimizer.step()
    scheduler.step()
for manual, pytorch_lr in zip(manual_lrs, pytorch_lrs):
    assert abs(manual - pytorch_lr) < 1e-6, f"manual {manual} vs LambdaLR {pytorch_lr}"
print(f"formula matches LambdaLR at every checkpoint: {[round(v,6) for v in pytorch_lrs]} ✓")
print("Exercise 4 passed ✓")

formula matches LambdaLR at every checkpoint: [0.0, 0.0005, 0.001, 0.00097, 0.0005, 0.0] ✓
Exercise 4 passed ✓


## Exercise 5 — Global Norm Clipping, By Hand

Implement `manual_clip_grad_norm_(parameters, max_norm)` per notes §7: compute ONE global norm across every parameter's gradient, then rescale every gradient in place by the same ratio if (and only if) that norm exceeds `max_norm`. Match `torch.nn.utils.clip_grad_norm_` exactly.

**Decision you're practicing:** clipping is a *global* norm across all parameters, not a per-tensor cap — and it only ever shrinks, never grows.

In [10]:
def manual_clip_grad_norm_(parameters, max_norm):
    """In-place global-norm clipping. Returns the ORIGINAL (pre-clip) global norm."""
    parameters = [p for p in parameters if p.grad is not None]
    global_norm = torch.sqrt(sum(p.grad.pow(2).sum() for p in parameters))   # ONE norm over everything
    scale = min(1.0, max_norm / (global_norm.item() + 1e-12))
    for p in parameters:
        p.grad.mul_(scale)                                                  # in-place, direction preserved
    return global_norm.item()

demo_params = [torch.zeros(2), torch.zeros(1)]
demo_params[0].grad = torch.tensor([3.0, 4.0])
demo_params[1].grad = torch.tensor([0.0])
norm = manual_clip_grad_norm_(demo_params, max_norm=2.0)
print(f"original norm: {norm}   (expect 5.0, from 3-4-5 triangle)")
print(f"clipped grad0: {demo_params[0].grad.tolist()}   (expect [1.2, 1.6])")
print(f"new norm: {(demo_params[0].grad.norm()**2 + demo_params[1].grad.norm()**2).sqrt().item():.4f}   (expect 2.0)")

original norm: 5.0   (expect 5.0, from 3-4-5 triangle)
clipped grad0: [1.2000000476837158, 1.600000023841858]   (expect [1.2, 1.6])
new norm: 2.0000   (expect 2.0)


**Verification**

In [11]:
# --- Verification: Exercise 5 ---
# case 1: over the cap -> must clip down to exactly max_norm
params_a = [torch.zeros(2)]
params_a[0].grad = torch.tensor([3.0, 4.0])          # norm = 5.0
params_b = [torch.zeros(2)]
params_b[0].grad = torch.tensor([3.0, 4.0])

reported_norm = manual_clip_grad_norm_(params_a, max_norm=2.0)
official_norm = torch.nn.utils.clip_grad_norm_(params_b, max_norm=2.0)
assert reported_norm is not None, "implement manual_clip_grad_norm_ first"
assert abs(reported_norm - 5.0) < 1e-5, f"original norm should be 5.0, got {reported_norm}"
assert torch.allclose(params_a[0].grad, params_b[0].grad, atol=1e-5), \
    f"clipped grad differs from torch's: {params_a[0].grad} vs {params_b[0].grad}"
new_norm = params_a[0].grad.norm().item()
assert abs(new_norm - 2.0) < 1e-4, f"clipped norm should be exactly max_norm=2.0, got {new_norm}"

# case 2: under the cap -> must NOT change anything
params_c = [torch.zeros(2)]
params_c[0].grad = torch.tensor([0.3, 0.4])          # norm = 0.5, well under max_norm=2.0
original = params_c[0].grad.clone()
manual_clip_grad_norm_(params_c, max_norm=2.0)
assert torch.equal(params_c[0].grad, original), "gradients under the cap must be left untouched"
print(f"over-cap clipped to exactly max_norm; under-cap left untouched ✓")
print("Exercise 5 passed ✓")

over-cap clipped to exactly max_norm; under-cap left untouched ✓
Exercise 5 passed ✓


## Exercise 6 — Accumulation With Clip/Step/Schedule on the Right Cadence

Implement `train_with_accumulation`: run `accum_steps` micro-batches per real step, and place `clip_grad_norm_`, `optimizer.step()`, `scheduler.step()`, and `zero_grad()` **only** on the real-step boundary (never per micro-batch). Verify the final weights exactly match one big-batch step (ch03's identity, now with clipping and a scheduler in the mix).

**Decision you're practicing:** a "step" spanning several micro-batches still gets exactly one clip/step/schedule/zero_grad, after the LAST micro-batch's backward.

In [12]:
def train_with_accumulation(model, full_inputs, full_targets, optimizer, scheduler, accum_steps, max_norm):
    """One real step built from accum_steps micro-batches. Mutates model/optimizer in place."""
    micro_inputs = full_inputs.split(len(full_inputs) // accum_steps)
    micro_targets = full_targets.split(len(full_targets) // accum_steps)

    optimizer.zero_grad(set_to_none=True)                    # ONCE, before any micro-batch
    for m_inputs, m_targets in zip(micro_inputs, micro_targets):
        loss = F.mse_loss(model(m_inputs), m_targets) / accum_steps
        loss.backward()                                       # accumulates across micro-batches

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)   # ONCE, on the FULL sum
    optimizer.step()                                          # ONCE
    scheduler.step()                                          # ONCE

torch.manual_seed(2)
full_inputs = torch.randn(8, 3)
full_targets = torch.randn(8, 1)

# the accumulated version
model_accum = nn.Linear(3, 1)
opt_accum = torch.optim.SGD(model_accum.parameters(), lr=0.1)
sched_accum = torch.optim.lr_scheduler.LambdaLR(opt_accum, lr_lambda=lambda s: 1.0)
train_with_accumulation(model_accum, full_inputs, full_targets, opt_accum, sched_accum,
                        accum_steps=4, max_norm=100.0)   # large max_norm so clipping is a no-op here

# the full-batch reference, same clip/step, no micro-batching
torch.manual_seed(2)
model_full = nn.Linear(3, 1)
opt_full = torch.optim.SGD(model_full.parameters(), lr=0.1)
opt_full.zero_grad(set_to_none=True)
F.mse_loss(model_full(full_inputs), full_targets).backward()
torch.nn.utils.clip_grad_norm_(model_full.parameters(), max_norm=100.0)
opt_full.step()

print(f"accumulated weight: {model_accum.weight.tolist()}")
print(f"full-batch weight : {model_full.weight.tolist()}")

accumulated weight: [[-0.06989515572786331, 0.40765872597694397, 0.048746127635240555]]
full-batch weight : [[0.11736783385276794, -0.02707013301551342, 0.062347304075956345]]


**Verification**

In [13]:
# --- Verification: Exercise 6 ---
torch.manual_seed(3)
full_inputs = torch.randn(8, 3)
full_targets = torch.randn(8, 1)

torch.manual_seed(5)
model_accum = nn.Linear(3, 1)
opt_accum = torch.optim.SGD(model_accum.parameters(), lr=0.1)
sched_accum = torch.optim.lr_scheduler.LambdaLR(opt_accum, lr_lambda=lambda s: 1.0)
train_with_accumulation(model_accum, full_inputs, full_targets, opt_accum, sched_accum,
                        accum_steps=4, max_norm=100.0)

torch.manual_seed(5)
model_full = nn.Linear(3, 1)
opt_full = torch.optim.SGD(model_full.parameters(), lr=0.1)
opt_full.zero_grad(set_to_none=True)
F.mse_loss(model_full(full_inputs), full_targets).backward()
torch.nn.utils.clip_grad_norm_(model_full.parameters(), max_norm=100.0)
opt_full.step()

assert torch.allclose(model_accum.weight, model_full.weight, atol=1e-5), \
    f"accumulated weight {model_accum.weight.tolist()} != full-batch {model_full.weight.tolist()}"
assert torch.allclose(model_accum.bias, model_full.bias, atol=1e-5)
assert sched_accum.last_epoch == 1, f"scheduler should have stepped exactly ONCE, stepped {sched_accum.last_epoch} times"

# a version that (wrongly) calls zero_grad per micro-batch must NOT match — sanity check the test itself
torch.manual_seed(5)
model_wrong = nn.Linear(3, 1)
opt_wrong = torch.optim.SGD(model_wrong.parameters(), lr=0.1)
for m_in, m_tgt in zip(full_inputs.split(2), full_targets.split(2)):
    opt_wrong.zero_grad(set_to_none=True)   # WRONG: clears accumulation every micro-batch
    (F.mse_loss(model_wrong(m_in), m_tgt) / 4).backward()
    opt_wrong.step()
assert not torch.allclose(model_wrong.weight, model_full.weight, atol=1e-4), \
    "sanity check failed: per-micro-batch zero_grad should NOT reproduce the full-batch step"
print("accumulated step matches full-batch exactly; scheduler ticked once, not per micro-batch ✓")
print("Exercise 6 passed ✓")

accumulated step matches full-batch exactly; scheduler ticked once, not per micro-batch ✓
Exercise 6 passed ✓


## Exercise 7 — The Checkpoint That Actually Resumes

Train a model with Adam for a few steps, save an **honest checkpoint** (model + optimizer + epoch), then resume into fresh objects and confirm training continues *without a stumble* — versus a **naive** model-only checkpoint, which visibly loses its momentum.

**Decision you're practicing:** why `optimizer.state_dict()` belongs in a checkpoint — Adam's `m_t`/`v_t` live there, not in the model.

In [14]:
def save_honest_checkpoint(model, optimizer, epoch, path):
    """Save model + optimizer + epoch."""
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "epoch": epoch}, path)

def load_honest_checkpoint(model, optimizer, path):
    """Load model + optimizer in place; return the saved epoch."""
    checkpoint = torch.load(path, map_location="cpu")
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    return checkpoint["epoch"]

import tempfile, os
CKPT_PATH = os.path.join(tempfile.gettempdir(), "b00_ch07_ckpt.pt")

torch.manual_seed(4)
model = nn.Linear(5, 5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.5)
probe = torch.randn(6, 5)
for _ in range(5):                                 # warm up Adam's momentum
    optimizer.zero_grad(set_to_none=True)
    model(probe).sum().backward()
    optimizer.step()
momentum_before_save = optimizer.state[model.weight]["exp_avg"].clone()
save_honest_checkpoint(model, optimizer, epoch=5, path=CKPT_PATH)
print(f"saved after epoch 5; a momentum buffer sample: {momentum_before_save.flatten()[:3].tolist()}")

saved after epoch 5; a momentum buffer sample: [0.2841523587703705, -0.5222273468971252, 1.5397450923919678]


**Verification**

In [15]:
# --- Verification: Exercise 7 ---
import tempfile, os
CKPT_PATH = os.path.join(tempfile.gettempdir(), "b00_ch07_ckpt.pt")

torch.manual_seed(6)
model = nn.Linear(5, 5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.5)
probe = torch.randn(6, 5)
for _ in range(5):
    optimizer.zero_grad(set_to_none=True)
    model(probe).sum().backward()
    optimizer.step()
momentum_before = optimizer.state[model.weight]["exp_avg"].clone()
save_honest_checkpoint(model, optimizer, epoch=5, path=CKPT_PATH)
assert os.path.exists(CKPT_PATH), "save_honest_checkpoint did not write a file"

# HONEST resume: momentum must survive exactly
resumed_model = nn.Linear(5, 5)
resumed_optimizer = torch.optim.Adam(resumed_model.parameters(), lr=0.5)
resumed_epoch = load_honest_checkpoint(resumed_model, resumed_optimizer, CKPT_PATH)
assert resumed_epoch == 5, f"epoch should resume as 5, got {resumed_epoch}"
resumed_momentum = resumed_optimizer.state[resumed_model.weight]["exp_avg"]
assert torch.equal(momentum_before, resumed_momentum), "honest checkpoint must preserve optimizer momentum exactly"
assert torch.equal(model.weight, resumed_model.weight), "weights must match too"

# NAIVE resume (weights only) must NOT have the momentum -- the bug this exercise demonstrates
naive_model = nn.Linear(5, 5)
naive_model.load_state_dict(torch.load(CKPT_PATH, map_location="cpu")["model"])
naive_optimizer = torch.optim.Adam(naive_model.parameters(), lr=0.5)   # fresh optimizer state
assert naive_model.weight not in naive_optimizer.state or not naive_optimizer.state[naive_model.weight], \
    "sanity: a freshly constructed optimizer should have NO momentum state yet"
print("honest checkpoint preserves optimizer momentum exactly; naive (model-only) resume loses it ✓")
print("Exercise 7 passed ✓")

honest checkpoint preserves optimizer momentum exactly; naive (model-only) resume loses it ✓
Exercise 7 passed ✓


## Exercise 8 — bf16 Autocast: Free Speedup, No Scaler

Run the same forward pass with and without `torch.autocast(dtype=torch.bfloat16)`, confirm the matmul runs in bf16 *inside* the block and fp32 outside, and prove a score of 70,000 (chapter 1's fp16-overflow example) stays finite in bf16 with **no `GradScaler`** anywhere in sight.

**Decision you're practicing:** bf16 autocast is a two-line wrap with none of fp16's scaler machinery, because bf16 keeps fp32's exponent range (chapter 1).

In [16]:
def run_with_autocast(model, inputs, use_bf16):
    """Run model(inputs) inside torch.autocast(dtype=bfloat16) if use_bf16, else plainly.
    Return the output tensor (so its .dtype can be inspected)."""
    if use_bf16:
        with torch.autocast(device_type=inputs.device.type, dtype=torch.bfloat16):
            return model(inputs)
    return model(inputs)

probe_model = nn.Linear(4, 4)
probe_inputs = torch.randn(2, 4)
plain_output = run_with_autocast(probe_model, probe_inputs, use_bf16=False)
autocast_output = run_with_autocast(probe_model, probe_inputs, use_bf16=True)
print(f"plain dtype   : {plain_output.dtype}")
print(f"autocast dtype: {autocast_output.dtype}   ← no GradScaler needed anywhere")

# the ch01 fp16-overflow number, surviving in bf16
big_score = torch.tensor(70000.0)
print(f"\n70000.0 in fp16: {big_score.to(torch.float16).item()}   (chapter 1's overflow)")
print(f"70000.0 in bf16: {big_score.to(torch.bfloat16).item()}   (finite — this is WHY no scaler is needed)")

plain dtype   : torch.float32
autocast dtype: torch.bfloat16   ← no GradScaler needed anywhere

70000.0 in fp16: inf   (chapter 1's overflow)
70000.0 in bf16: 70144.0   (finite — this is WHY no scaler is needed)


**Verification**

In [17]:
# --- Verification: Exercise 8 ---
probe_model = nn.Linear(4, 4)
probe_inputs = torch.randn(2, 4)
plain_output = run_with_autocast(probe_model, probe_inputs, use_bf16=False)
autocast_output = run_with_autocast(probe_model, probe_inputs, use_bf16=True)
assert plain_output is not None and autocast_output is not None, "implement run_with_autocast first"
assert plain_output.dtype == torch.float32, f"plain run should stay float32, got {plain_output.dtype}"
assert autocast_output.dtype == torch.bfloat16, f"autocast run should be bfloat16, got {autocast_output.dtype}"

# the range argument: bf16 must keep 70000 finite where fp16 would not
big_score = torch.tensor(70000.0)
assert torch.isinf(big_score.to(torch.float16)), "sanity: fp16 should overflow on 70000 (ch01)"
assert torch.isfinite(big_score.to(torch.bfloat16)), "bf16 must stay finite on 70000 — the whole point of autocast needing no scaler"
print("autocast produces bf16 activations; bf16 stays finite where fp16 overflows — no GradScaler required ✓")
print("Exercise 8 passed ✓")

autocast produces bf16 activations; bf16 stays finite where fp16 overflows — no GradScaler required ✓
Exercise 8 passed ✓


---
## Done!

Compare your work against `solved/ch07-training-loop-anatomy-solved.ipynb`.

The **training block** (ch06–ch07) is complete: batches in, a correctly-ordered loop driving them, optimizers and schedules tuned, gradients clipped, checkpoints that actually resume. Next is the **operations block** — **ch08 — Devices, Checkpoints & Debugging**: reading shape-error tracebacks, hunting NaNs, `state_dict` surgery, and GPU memory basics.